# 01 — Hair Conditioners Taxonomy V1: Historical Review and V3 Readiness

## Objectives

Audit the approved Taxonomy V1 as a historical, qualitative human-review artifact and verify that the corrected V3 discovery sample and batch plan are ready for a new external run. The notebook never calls Gemini, never treats legacy weighted support as prevalence, and never presents Taxonomy V2 as completed.

## Цели

Проверить утверждённую Taxonomy V1 как исторический качественный результат ручного ревью и подтвердить готовность исправленной выборки и batch plan V3 к новому внешнему запуску. Ноутбук не обращается к Gemini, не интерпретирует старую взвешенную поддержку как распространённость и не выдаёт Taxonomy V2 за завершённую.

In [ ]:
# Standard library and project discovery / Стандартная библиотека и поиск проекта
import hashlib
import json
import sys
from pathlib import Path 

# Analysis and notebook display / Аналитика и отображение в ноутбуке
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reusable project path helper / Переиспользуемая функция пути проекта
from src.common.project import find_project_root

In [ ]:
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
DATASET_VERSION = "amazon_reviews_2023_beauty_2021_2023_v1"
TAXONOMY_VERSION = "hair_conditioners_taxonomy_v1"
DISCOVERY_VERSION = "hair_conditioners_discovery_v3"
PROPOSED_TAXONOMY_VERSION = "hair_conditioners_taxonomy_v2"

TAXONOMY_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_taxonomy_v1.json"
REVIEWED_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_taxonomy_v1_reviewed.csv"
GAP_AUDIT_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_taxonomy_v1_gap_audit.md"
VALIDITY_PATH = PROJECT_ROOT / "reports/aspects/artifact_validity_v1.json"
DISCOVERY_CONFIG_PATH = PROJECT_ROOT / "config/aspects/hair_conditioners_discovery_v3.json"
SAMPLE_REPORT_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_v2_sample_report.json"
PLAN_REPORT_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_discovery_v3_batch_plan.json"
READINESS_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_discovery_v3_gemini_readiness.json"
SAMPLE_PATH = PROJECT_ROOT / "data/processed" / DATASET_VERSION / "aspect_discovery/hair_conditioners_v2_sample.parquet"
PLAN_PATH = PROJECT_ROOT / "data/processed" / DATASET_VERSION / "aspect_discovery/hair_conditioners_discovery_v3/batch_plan.jsonl"
RESPONSES_PATH = PLAN_PATH.parent / "responses"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 1. Establish artifact lineage and validity

Taxonomy V1 remains approved for its human-reviewed labels, definitions, aliases, and group structure. Its legacy weighted-support fields are explicitly invalid for population inference because the old sampler applied a product cap before assigning review-level weights. Corrected inference requires the V3 discovery run and a separately reviewed Taxonomy V2.

## 1. Проверка происхождения и валидности артефактов

Taxonomy V1 остаётся утверждённой для проверенных человеком названий, определений, синонимов и структуры групп. Её старые поля взвешенной поддержки нельзя использовать для выводов о генеральной совокупности: прежний отбор ограничивал число отзывов товара до расчёта review-level весов. Для корректных оценок нужны запуск discovery V3 и отдельное ручное ревью Taxonomy V2.

In [ ]:
required_paths = [
    TAXONOMY_PATH,
    REVIEWED_PATH,
    GAP_AUDIT_PATH,
    VALIDITY_PATH,
    DISCOVERY_CONFIG_PATH,
    SAMPLE_REPORT_PATH,
    PLAN_REPORT_PATH,
    READINESS_PATH,
    SAMPLE_PATH,
    PLAN_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing aspect-review inputs: {missing_paths}"

taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
reviewed = pd.read_csv(REVIEWED_PATH, keep_default_na=False)
validity = json.loads(VALIDITY_PATH.read_text(encoding="utf-8"))
discovery_config = json.loads(DISCOVERY_CONFIG_PATH.read_text(encoding="utf-8"))
sample_report = json.loads(SAMPLE_REPORT_PATH.read_text(encoding="utf-8"))
plan_report = json.loads(PLAN_REPORT_PATH.read_text(encoding="utf-8"))
readiness = json.loads(READINESS_PATH.read_text(encoding="utf-8"))
sample = pd.read_parquet(SAMPLE_PATH)

assert taxonomy["status"] == "approved"
assert taxonomy["taxonomy_version"] == TAXONOMY_VERSION
assert taxonomy["dataset_version"] == DATASET_VERSION
assert taxonomy["aspect_count"] == 33
assert discovery_config["discovery_version"] == DISCOVERY_VERSION
assert discovery_config["proposed_taxonomy_version"] == PROPOSED_TAXONOMY_VERSION
assert discovery_config["dataset_version"] == DATASET_VERSION
assert plan_report["discovery_version"] == DISCOVERY_VERSION
assert plan_report["sample_schema_version"] == discovery_config["sample_schema_version"]
assert plan_report["sample_review_count"] == len(sample) == 1500
assert plan_report["batch_count"] == 15
assert plan_report["sample_sha256"] == sha256_file(SAMPLE_PATH)
assert plan_report["plan_sha256"] == sha256_file(PLAN_PATH)
assert sample_report["sampling_design"] == "stratified_simple_random_sampling_without_replacement"
assert sample_report["product_cap_applied"] is False
assert abs(sample["sampling_weight"].sum() - sample_report["eligible_review_count"]) < 1e-6

legacy_entry = next(
    item for item in validity["artifacts"]
    if item["path"] == "reports/aspects/hair_conditioners_taxonomy_v1.json"
)
corrected_sample = validity["corrected_artifacts"]["discovery_sample"]
corrected_plan = validity["corrected_artifacts"]["discovery_plan"]
assert legacy_entry["status"] == "approved_qualitative_taxonomy"
assert legacy_entry["labels_and_aliases_valid"] is True
assert legacy_entry["weighted_support_fields_valid"] is False
assert corrected_sample["status"] == "ready"
assert corrected_sample["sha256"] == sha256_file(SAMPLE_PATH)
assert corrected_plan["status"] == "awaiting_external_gemini_responses"

response_count = len(list(RESPONSES_PATH.glob("*.json"))) if RESPONSES_PATH.is_dir() else 0
readiness_summary = pd.Series(
    {
        "historical_taxonomy": TAXONOMY_VERSION,
        "historical_taxonomy_role": "qualitative labels and aliases only",
        "legacy_weighted_prevalence_allowed": False,
        "corrected_sample_design": sample_report["sampling_design"],
        "corrected_sample_reviews": len(sample),
        "corrected_sample_weight_total": sample["sampling_weight"].sum(),
        "v3_batch_count": plan_report["batch_count"],
        "gemini_client_ready": readiness["ready_for_client_initialization"],
        "saved_v3_responses": response_count,
        "v3_status": corrected_plan["status"],
        "proposed_next_taxonomy": PROPOSED_TAXONOMY_VERSION,
    },
    name="value",
)
display(readiness_summary.to_frame())

## 2. Review only qualitative taxonomy content

The tables below intentionally exclude legacy mention counts, weighted support, weighted population shares, and rank-by-prevalence views. They describe the semantic inventory and the recorded human decisions, not how common an aspect is among all eligible reviews.

## 2. Анализ только качественного содержания таксономии

Из таблиц намеренно исключены старые числа упоминаний, взвешенная поддержка, доли генеральной совокупности и рейтинги по распространённости. Здесь показаны семантический состав и зафиксированные решения человека, а не частота аспектов среди всех подходящих отзывов.

In [ ]:
prohibited_prevalence_fields = {
    "mention_count",
    "sample_review_count",
    "product_count",
    "weighted_review_support",
    "weighted_population_share",
    "weighted_sample_population",
}
qualitative_columns = [
    "aspect_id",
    "canonical_name",
    "parent_group",
    "definition",
    "aliases",
    "source_candidate_keys",
]
assert prohibited_prevalence_fields.isdisjoint(qualitative_columns)

aspects = pd.DataFrame(taxonomy["aspects"])
qualitative_aspects = aspects[qualitative_columns].copy()
qualitative_aspects["aliases"] = qualitative_aspects["aliases"].map(" | ".join)
qualitative_aspects["source_candidate_keys"] = qualitative_aspects["source_candidate_keys"].map(" | ".join)
assert qualitative_aspects["aspect_id"].is_unique
assert len(qualitative_aspects) == taxonomy["aspect_count"]

assert len(reviewed) == 23
assert reviewed["aspect_id"].is_unique
assert set(reviewed["review_decision"]) == {"approve", "edit", "split", "exclude"}
assert reviewed["review_decision"].value_counts().to_dict() == taxonomy["review_decision_counts"]

decision_summary = (
    reviewed["review_decision"]
    .value_counts()
    .rename_axis("review_decision")
    .reset_index(name="proposal_rows")
)
group_summary = (
    qualitative_aspects.groupby("parent_group", as_index=False)
    .agg(aspect_count=("aspect_id", "count"))
    .sort_values(["aspect_count", "parent_group"], ascending=[False, True])
)

display(decision_summary)
display(group_summary)
display(qualitative_aspects)

plot_data = group_summary.sort_values("aspect_count")
figure, axis = plt.subplots(figsize=(9, 5))
axis.barh(plot_data["parent_group"], plot_data["aspect_count"], color="#4C78A8")
axis.set_title("Taxonomy V1 semantic structure (not prevalence)")
axis.set_xlabel("Approved qualitative aspect count")
axis.set_ylabel("Parent group")
axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
plt.show()

## 3. Preserve the human decision trail and open gaps

The historical review converted 23 proposal rows into 33 approved qualitative aspects through approvals, edits, exclusions, and semantic splits. The gap audit records what was resolved in V1 and what still requires evidence from V3; it does not retrofit new frequency estimates onto the old extraction.

## 3. Сохранение истории ручных решений и открытых пробелов

Историческое ревью преобразовало 23 строки предложения в 33 утверждённых качественных аспекта через подтверждения, правки, исключения и смысловые разделения. Gap audit фиксирует, что решено в V1 и что ещё требует данных V3; новые оценки частоты не приписываются старому извлечению задним числом.

In [ ]:
decision_columns = [
    "aspect_id",
    "canonical_name",
    "review_decision",
    "approved_name",
    "approved_parent_group",
    "review_notes",
]
assert prohibited_prevalence_fields.isdisjoint(decision_columns)
display(reviewed[decision_columns])
display(Markdown(GAP_AUDIT_PATH.read_text(encoding="utf-8")))

## Conclusion

Taxonomy V1 is retained as a 33-aspect qualitative, human-reviewed vocabulary; none of its legacy weighted-support fields is used as a prevalence estimate. The corrected 1,500-review SRS sample and 15-batch V3 plan pass identity and SHA-256 checks. The plan is still awaiting external Gemini responses, so Taxonomy V2 does not yet exist and no new human review is claimed.

## Вывод

Taxonomy V1 сохраняется как качественный словарь из 33 аспектов, проверенный человеком; ни одно старое поле взвешенной поддержки не используется как оценка распространённости. Исправленная SRS-выборка из 1 500 отзывов и план V3 из 15 батчей прошли проверки identity и SHA-256. План всё ещё ожидает внешних ответов Gemini, поэтому Taxonomy V2 пока не создана и новое ручное ревью не заявляется.